# 🐛 Critterboard — Persona LoRA Training (Kaggle T4)

**Run this AFTER you've curated examples locally.** Upload `data/curated/larva.jsonl`, `data/curated/snail.jsonl`, `data/curated/maywind.jsonl` as a Kaggle Dataset and attach it to this notebook.

What this notebook does:
1. Installs PEFT + TRL + transformers
2. Loads `meta-llama/Llama-3.2-1B-Instruct` (request access on Hugging Face first)
3. Trains one LoRA adapter per persona
4. Saves them to `/kaggle/working/adapters/<persona>/`

**Runtime:** Enable `Settings → Accelerator → GPU T4 x2`  
**Expected duration:** ~30-50 minutes for all three personas at 300 examples each  
**Internet:** turn ON (needed to fetch the base weights)

Once it finishes, download the `adapters/` folder from the Output panel and run `05_export_adapters.py` locally to produce the GGUF files.

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate sentencepiece

In [ ]:
import os, json, torch
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

# Your HF token (request access to Llama 3.2 first)
from huggingface_hub import login
if 'HF_TOKEN' in os.environ:
    login(token=os.environ['HF_TOKEN'])

BASE = 'meta-llama/Llama-3.2-1B-Instruct'
INPUT_DIR  = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/adapters'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Persona system prompts — must match the strings in src/personas/index.ts
PROMPTS = {
    'larva':   'You are Prof. Larva, a deadpan, sarcastic but ultimately helpful AI bug expert in an offline insect-ID app. Reply in 1–2 short sentences, sardonic but factually correct. No emoji, no quotes.',
    'snail':   'You are Dr. Snail, a patient, methodical, calm AI naturalist in an offline insect-ID app. Reply in 1–2 short sentences, warm and reassuring, gently educational. No emoji, no quotes. Always take your time.',
    'maywind': 'You are R.A. Maywind, a witty, joyful, happy-go-lucky AI research assistant in an offline insect-ID app. Reply in 1–2 short sentences, enthusiastic, curious, light-hearted. No emoji, no quotes.',
}
print('OK')

In [ ]:
# Find the dataset you uploaded.  Default expectation: a Kaggle Dataset
# named 'critterboard-personas' with the three .jsonl files at its root.
candidates = list(INPUT_DIR.rglob('*.jsonl'))
print('Found JSONL files:'); [print(' ', p) for p in candidates]

In [ ]:
def load_dataset_for(persona):
    matches = [p for p in candidates if p.stem == persona]
    if not matches: raise FileNotFoundError(f'No curated file for {persona}')
    rows = [json.loads(l) for l in matches[0].read_text().splitlines() if l.strip()]
    formatted = [{
        'messages': [
            {'role': 'system',    'content': PROMPTS[persona]},
            {'role': 'user',      'content': r['instruction']},
            {'role': 'assistant', 'content': r['output']},
        ]
    } for r in rows]
    return Dataset.from_list(formatted), len(rows)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def train_one(persona):
    print(f'\n══ Training {persona} ══')
    ds, n = load_dataset_for(persona)
    print(f'  {n} examples')

    # Fresh base each time — keeps adapters independent.
    base = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=torch.bfloat16, device_map='auto'
    )
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    )
    model = get_peft_model(base, lora)
    model.print_trainable_parameters()

    out = OUTPUT_DIR / persona
    args = TrainingArguments(
        output_dir=str(out),
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=1,
        bf16=True,
        report_to='none',
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, args=args,
        train_dataset=ds, max_seq_length=512,
    )
    trainer.train()
    trainer.save_model(str(out))
    del trainer, model, base
    torch.cuda.empty_cache()
    print(f'  ✓ saved → {out}')

In [ ]:
for persona in ['larva', 'snail', 'maywind']:
    train_one(persona)

print('\nDone. Download /kaggle/working/adapters/ from the Output panel.')
print('Then locally:')
print('  python 05_export_adapters.py --llama-cpp ~/code/llama.cpp')